In [ ]:

!pip install pandas openpyxl --quiet

In [ ]:

import re
import unicodedata
import pandas as pd
from collections import Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

INPUT_FILE          = 'Question4.xlsx'
OUTPUT_FILE         = 'Q4_Lattice_WER_Results.xlsx'
SHEET_NAME          = 'Task'
MODELS              = ['Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n']
MODEL_I             = 'Model i'   
CONSENSUS_THRESHOLD = 3           

NORMALIZE_MAP = {
    # Chandrabindu -> Anusvar variants (same pronunciation, different Unicode)
    'हाँ':'हां', 'वहाँ':'वहां', 'यहाँ':'यहां', 'वहँा':'वहां',
    'बनाऊँगी':'बनाऊंगी', 'रखूँ':'रखूं', 'हूँ':'हूं',
    # Nukta variants
    'सब्ज़ी':'सब्जी', 'ज़्यादा':'ज्यादा',
    # Roman loanwords -> Devanagari (Model k writes English in Roman script)
    'feedback':'फीडबैक', 'pure':'प्योर', 'heart':'हार्ट', 'face':'फेस',
    'desktop':'डेस्कटॉप', 'laptop':'लैपटॉप', 'easy':'इजी', 'sir':'सर',
}

# Proper nouns: ASR is unreliable here. Require Model i confirmation. (Rule 3)
PROPER_NOUNS = {
    'सियाराम', 'राम', 'कृष्ण', 'गणेश', 'रक्षाबंधन',
    'कोटा', 'राजस्थान', 'प्रयागराज', 'गणेशजी',
}

# Filler/disfluency words. If >= 5 models delete them, forgive it. (Rule 5)
FILLER_WORDS = {
    'हम्म', 'हम', 'उम', 'ओम', 'अ', 'आ', 'एह',
    'मतलब', 'देखो', 'सुनो', 'यानी', 'यानि',
}

# Rank-based model weights (balanced — no single model dominates)
# Model i = 0.286, Model H = 0.238, ... Model m = 0.048
_RANKS        = {'Model i':6,'Model H':5,'Model k':4,'Model l':3,'Model n':2,'Model m':1}
MODEL_WEIGHTS = {m: round(r/sum(_RANKS.values()),3) for m,r in _RANKS.items()}

print('Configuration loaded.')
print(f'  Models  : {MODELS}')
print(f'  Weights : {MODEL_WEIGHTS}')

Configuration loaded.
  Models  : ['Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n']
  Weights : {'Model i': 0.286, 'Model H': 0.238, 'Model k': 0.19, 'Model l': 0.143, 'Model n': 0.095, 'Model m': 0.048}


In [ ]:
#  Preprocessing ─────────────────────────────────────────────────────
PUNCT_RE = re.compile(r'[।॥,\.?!;:\-–—…"\' ()\[\]]+')

def preprocess(text: str) -> list:
    """Strip punctuation, NFC-normalize, apply spelling equivalences."""
    if not isinstance(text, str) or text.strip() in ('', 'nan', '--'):
        return []
    text = unicodedata.normalize('NFC', text)
    text = PUNCT_RE.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [unicodedata.normalize('NFC', w.strip().lower())
             for w in text.split() if w.strip()]
    return [NORMALIZE_MAP.get(w, w) for w in words if w]


# Load dataset
df_raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
df_raw = df_raw[[c for c in df_raw.columns if not c.startswith('Unnamed')]]
print(f'Loaded {len(df_raw)} segments, columns: {list(df_raw.columns)}')


test_cases = [
    'वही अपना खेती बाड़ी और क्या',
    'हाँ और यहाँ पर सारी सुविधा है।',
    'जी feedback मिलने पर सुधार करना।',
]
print('\nPreprocessing examples:')
for t in test_cases:
    print(f'  IN : {t}')
    print(f'  OUT: {preprocess(t)}')
    print()

Loaded 46 segments, columns: ['segment_url_link', 'Human', 'Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n']

Preprocessing examples:
  IN : वही अपना खेती बाड़ी और क्या
  OUT: ['वही', 'अपना', 'खेती', 'बाड़ी', 'और', 'क्या']

  IN : हाँ और यहाँ पर सारी सुविधा है।
  OUT: ['हां', 'और', 'यहां', 'पर', 'सारी', 'सुविधा', 'है']

  IN : जी feedback मिलने पर सुधार करना।
  OUT: ['जी', 'फीडबैक', 'मिलने', 'पर', 'सुधार', 'करना']



In [ ]:
#  algorithms 

def levenshtein(a: str, b: str) -> int:
    """Character-level edit distance."""
    if a == b: return 0
    la, lb = len(a), len(b)
    if la == 0: return lb
    if lb == 0: return la
    prev = list(range(lb + 1))
    for ca in a:
        curr = [prev[0] + 1]
        for j, cb in enumerate(b):
            curr.append(min(prev[j]+(ca!=cb), curr[j]+1, prev[j+1]+1))
        prev = curr
    return prev[lb]


def dp_align(ref: list, hyp: list) -> list:
    """
    Word-level DP alignment of hypothesis to reference.
    Returns list of (ref_word_or_None, hyp_word_or_None, operation).
    Operations: 'match', 'sub', 'del', 'ins'
    """
    n, m = len(ref), len(hyp)
    dp = [[float('inf')]*(m+1) for _ in range(n+1)]
    dp[0][0] = 0
    for i in range(1,n+1): dp[i][0] = i
    for j in range(1,m+1): dp[0][j] = j
    for i in range(1,n+1):
        for j in range(1,m+1):
            c = 0 if ref[i-1]==hyp[j-1] else 1
            dp[i][j] = min(dp[i-1][j-1]+c, dp[i-1][j]+1, dp[i][j-1]+1)
    al, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            c = 0 if ref[i-1]==hyp[j-1] else 1
            if dp[i][j] == dp[i-1][j-1]+c:
                al.append((ref[i-1], hyp[j-1], 'match' if c==0 else 'sub'))
                i -= 1; j -= 1; continue
        if i > 0 and (j==0 or dp[i][j]==dp[i-1][j]+1):
            al.append((ref[i-1], None, 'del')); i -= 1
        else:
            al.append((None, hyp[j-1], 'ins')); j -= 1
    al.reverse()
    return al


def get_script(word: str) -> str:
    """Detect primary Unicode script."""
    for ch in word:
        name = unicodedata.name(ch, '')
        if 'DEVANAGARI' in name: return 'dev'
        if 'ARABIC'    in name: return 'ara'
        if 'LATIN'     in name: return 'lat'
    return 'oth'


# Quick verification
print('levenshtein:')
print(f'  वेव्स/वेब्स = {levenshtein("वेव्स","वेब्स")}  (expect 1)')
print(f'  एक/ठीक      = {levenshtein("एक","ठीक")}  (expect 2)')
print()
print('get_script:')
for w in ['मेहनत','مेحنت','feedback']:
    print(f'  {w} -> {get_script(w)}')
print()
print('dp_align:')
r = preprocess('मौनता का अर्थ क्या होता है')
h = preprocess('मौन तागार थके होतई')
al = dp_align(r, h)
for row in al:
    print(f'  {row}')

levenshtein:
  वेव्स/वेब्स = 1  (expect 1)
  एक/ठीक      = 2  (expect 2)

get_script:
  मेहनत -> dev
  مेحنت -> ara
  feedback -> lat

dp_align:
  ('मौनता', None, 'del')
  ('का', None, 'del')
  ('अर्थ', 'मौन', 'sub')
  ('क्या', 'तागार', 'sub')
  ('होता', 'थके', 'sub')
  ('है', 'होतई', 'sub')


In [ ]:
# Consensus insertions 
# When >= CONSENSUS_THRESHOLD models insert the same word at the same position,
# the reference is probably MISSING that word from the audio.

def find_consensus_insertions(ref: list, model_outputs: dict,
                               threshold: int = CONSENSUS_THRESHOLD) -> dict:
    """
    Returns {ref_position: set_of_words} for words threshold+ models insert.
    """
    vote_counter = Counter()
    for m, hyp in model_outputs.items():
        al = dp_align(ref, hyp)
        ref_pos, seen = 0, set()
        for (_, hw, op) in al:
            if op in ('match','sub','del'):
                ref_pos += 1
            elif op == 'ins':
                key = (ref_pos, hw)
                if key not in seen:
                    vote_counter[key] += 1
                    seen.add(key)
    consensus = {}
    for (pos, word), count in vote_counter.items():
        if count >= threshold:
            consensus.setdefault(pos, set()).add(word)
    return consensus


# Verify on known cases: segs 13, 26, 37 have ref missing a word
for seg_num, expected in [(13,'तो'), (26,'है'), (37,'साथ')]:
    row  = df_raw.iloc[seg_num-1]
    ref  = preprocess(str(row['Human']))
    outs = {m: preprocess(str(row[m])) for m in MODELS}
    ci   = find_consensus_insertions(ref, outs)
    found = any(expected in words for words in ci.values())
    print(f"Seg {seg_num}: consensus_inserts={ci}")
    print(f"  '{expected}' detected: {found}  <- ref was missing this word")
    print()

Seg 13: consensus_inserts={}
  'तो' detected: False  <- ref was missing this word

Seg 26: consensus_inserts={}
  'है' detected: False  <- ref was missing this word

Seg 37: consensus_inserts={}
  'साथ' detected: False  <- ref was missing this word



In [ ]:
# ── trust_decision() — 8 rules 
#
# Rule 1   edit<=1 orthographic variant          -> always add to bin
# Arabic   Arabic/Urdu script                    -> always reject
# Latin    Roman loanword                        -> accept if >=2 models agree
# Rule 2A  Model i agrees with alt               -> trust models
# Rule 2B  weighted score >0.45 + edit<=2        -> trust models
# Rule 2C  weighted score >0.45, Model i disagrees -> trust reference
# Rule 2D  weighted score <=0.45                 -> trust reference
# Rule 3   proper noun, Model i confirms ref     -> trust reference

def trust_decision(ref_word, alt_word, alt_models, all_model_tokens,
                   model_i_token, model_outputs):
    """Returns (should_add: bool, rule_applied: str)"""
    if not alt_word or alt_word == ref_word:
        return True, 'identical'

    if get_script(alt_word) == 'ara':
        return False, 'arabic_rejected'

    if get_script(alt_word) == 'lat':
        sup = sum(1 for t in all_model_tokens if t == alt_word)
        return (True,'latin_consensus') if sup >= 2 else (False,'latin_no_support')

    edit  = levenshtein(ref_word, alt_word)
    w_alt = sum(MODEL_WEIGHTS.get(m,0) for m in alt_models)
    mi_alt = (model_i_token == alt_word)

    if edit <= 1:
        return True, 'rule1_orthographic'

    if ref_word in PROPER_NOUNS:
        all_others = (len(alt_models) >= len(model_outputs)-1)
        return (True,'rule3_proper_mi_agrees') if (mi_alt or all_others) \
               else (False,'rule3_proper_trust_ref')

    if mi_alt and len(alt_models) >= 2:
        return True, 'rule2a_mi_agrees'

    if w_alt > 0.45 and edit <= 2:
        return True, 'rule2b_weighted_plausible'

    if w_alt > 0.45 and not mi_alt:
        return False, 'rule2c_mi_disagrees'

    return False, 'rule2d_weighted_ref'


# Demo: every contested case from the data
demo = [
    ('सियाराम','आराम',  ['Model H','Model m','Model n'],        'सियाराम'),
    ('वेव्स',  'वेब्स', ['Model i','Model k','Model m'],        'वेब्स'),
    ('एक',     'ठीक',   ['Model H','Model k','Model m','Model n'],'एक'),
    ('मैं',    'में',   ['Model H','Model l','Model m'],        'मैं'),
    ('से',     'सबसे',  ['Model k','Model l','Model m','Model n'],'से'),
]
print(f'{"Ref":<14} {"Alt":<14} {"Decision":<16}  Rule')
print('-'*62)
for ref_w, alt_w, alt_ms, mi in demo:
    tokens = [alt_w]*len(alt_ms)
    ok, rule = trust_decision(ref_w, alt_w, alt_ms, tokens, mi, MODELS)
    print(f'{ref_w:<14} {alt_w:<14} {"ADD TO BIN" if ok else "TRUST REF ":<16}  {rule}')

Ref            Alt            Decision          Rule
--------------------------------------------------------------
सियाराम        आराम           TRUST REF         rule3_proper_trust_ref
वेव्स          वेब्स          ADD TO BIN        rule1_orthographic
एक             ठीक            ADD TO BIN        rule2b_weighted_plausible
मैं            में            ADD TO BIN        rule1_orthographic
से             सबसे           ADD TO BIN        rule2b_weighted_plausible


In [ ]:
# Cell 7: build_lattice() 

def build_lattice(ref: list, model_outputs: dict) -> tuple:
    """
    Construct word-level lattice bins.
    Returns (lattice_list, consensus_insertions_dict)
    Each bin: {position, ref_word, valid_alternatives, forgive_deletion, trust_log, model_words}
    """
    # Align each model to the reference
    ref_aligned = {}
    for m, hyp in model_outputs.items():
        al   = dp_align(ref, hyp)
        toks = [hw for (_,hw,op) in al if op in ('match','sub','del')]
        while len(toks) < len(ref): toks.append(None)
        ref_aligned[m] = toks[:len(ref)]

    # Consensus insertions — ref may be missing words
    consensus_inserts = find_consensus_insertions(ref, model_outputs)

    lattice = []
    for pos, ref_word in enumerate(ref):
        model_words  = {m: ref_aligned[m][pos] for m in model_outputs}
        model_tokens = [w for w in model_words.values() if w]
        model_i_tok  = model_words.get(MODEL_I)
        valid_alts   = {ref_word}
        trust_log    = {}

        # Compound word splits and merges
        if pos < len(ref)-1:
            comp = ref_word + ref[pos+1]
            if any(t==comp for t in model_tokens):
                valid_alts.add(comp); trust_log[comp]='compound_merge'
        for m, hyp in model_outputs.items():
            for k in range(len(hyp)-1):
                if hyp[k]+hyp[k+1] == ref_word:
                    valid_alts.add(hyp[k]);   trust_log[hyp[k]]  ='compound_split'
                    valid_alts.add(hyp[k+1]); trust_log[hyp[k+1]]='compound_split'

        # Apply trust_decision to every unique alternative
        for alt_word, _ in Counter(t for t in model_tokens if t and t!=ref_word).items():
            alt_models = [m for m in model_outputs if model_words.get(m)==alt_word]
            ok, rule   = trust_decision(ref_word, alt_word, alt_models,
                                        model_tokens, model_i_tok, model_outputs)
            if ok:
                valid_alts.add(alt_word); trust_log[alt_word]=rule

        # Deletion forgiveness
        votes_del    = sum(1 for m in model_outputs if model_words.get(m) is None)
        any_override = any('rule2' in v or 'rule3_proper_mi' in v
                           for v in trust_log.values())
        forgive_del  = (
            any_override or
            (ref_word in FILLER_WORDS and votes_del >= len(model_outputs)-1)
        )

        lattice.append({
            'position'          : pos,
            'ref_word'          : ref_word,
            'valid_alternatives': valid_alts,
            'forgive_deletion'  : forgive_del,
            'trust_log'         : trust_log,
            'model_words'       : model_words,
        })
    return lattice, consensus_inserts



row0  = df_raw.iloc[0]
ref0  = preprocess(str(row0['Human']))
outs0 = {m: preprocess(str(row0[m])) for m in MODELS}
lat0, ci0 = build_lattice(ref0, outs0)
print(f'Segment 1 Ref: "{row0["Human"]}"')
print(f'{"Pos":<5} {"Ref word":<20} Valid alternatives')
print('-'*55)
for b in lat0:
    extras = b['valid_alternatives'] - {b['ref_word']}
    fd = ' [DEL OK]' if b['forgive_deletion'] else ''
    print(f"{b['position']:<5} {b['ref_word']:<20} {extras or '(ref only)'}{fd}")

Segment 1 Ref: "वही अपना खेती बाड़ी और क्या"
Pos   Ref word             Valid alternatives
-------------------------------------------------------
0     वही                  (ref only)
1     अपना                 (ref only)
2     खेती                 (ref only)
3     बाड़ी                (ref only)
4     और                   (ref only)
5     क्या                 (ref only)


In [ ]:
# compute_wer() score one hypothesis against the lattice

def compute_wer(ref: list, hyp: list, lattice: list,
                consensus_inserts: dict) -> tuple:
    """
    Returns (std_wer, lattice_wer, detail_dict)
    SUB: not penalized if hyp token is in the bin's valid_alternatives
    DEL: not penalized if forgive_deletion is True
    INS: not penalized if the word is in consensus_inserts (ref was missing it)
    """
    if not ref:
        empty = {'S':0,'D':0,'I':0,'N':0,'lattice_S':0,'lattice_D':0,'lattice_I':0}
        return 0.0, 0.0, empty

    alignment = dp_align(ref, hyp)
    S=D=I=lattice_S=lattice_D=lattice_I=0
    N=len(ref); pos=0

    for (rw, hw, op) in alignment:
        if op == 'match':
            pos += 1

        elif op == 'sub':
            S += 1
            bin_  = lattice[pos] if pos<len(lattice) else {'valid_alternatives':{rw}}
            valid = bin_['valid_alternatives']
            if hw in valid: pass                                      # exact match
            elif any(levenshtein(hw,va)<=1 for va in valid if va): pass  # fuzzy
            else: lattice_S += 1
            pos += 1

        elif op == 'del':
            D += 1
            bin_ = lattice[pos] if pos<len(lattice) else {'forgive_deletion':False}
            if not bin_['forgive_deletion']: lattice_D += 1
            pos += 1

        elif op == 'ins':
            I += 1
            prev_pos  = max(0, pos-1)
            valid_ins = (consensus_inserts.get(pos,set()) |
                         consensus_inserts.get(prev_pos,set()))
            if hw not in valid_ins: lattice_I += 1

    std_w = (S+D+I)/N
    lat_w = (lattice_S+lattice_D+lattice_I)/N
    return std_w, lat_w, {
        'S':S,'D':D,'I':I,'N':N,
        'lattice_S':lattice_S,'lattice_D':lattice_D,'lattice_I':lattice_I
    }


# Demo on segment 2 (models k and m hallucinate badly)
row2  = df_raw.iloc[1]
ref2  = preprocess(str(row2['Human']))
outs2 = {m: preprocess(str(row2[m])) for m in MODELS}
lat2, ci2 = build_lattice(ref2, outs2)
print(f'Segment 2 Ref: "{row2["Human"]}"')
print(f'{"Model":<12} {"Std%":>8} {"Lat%":>8} {"Saved":>8}')
print('-'*42)
for m in MODELS:
    sw, lw, _ = compute_wer(ref2, outs2[m], lat2, ci2)
    print(f'{m:<12} {sw*100:>8.1f} {lw*100:>8.1f} {(sw-lw)*100:>8.1f}')

Segment 2 Ref: "मौनता का अर्थ क्या होता है"
Model            Std%     Lat%    Saved
------------------------------------------
Model H           0.0      0.0      0.0
Model i           0.0      0.0      0.0
Model k         100.0    100.0      0.0
Model l          33.3     16.7     16.7
Model m         100.0    100.0      0.0
Model n          33.3     33.3      0.0


In [ ]:
# run_evaluation() + final results 

def run_evaluation(df: pd.DataFrame) -> tuple:
    """Run standard + lattice WER over all segments and all models."""
    std_totals = {m:{'S':0,'D':0,'I':0,'N':0} for m in MODELS}
    lat_totals = {m:{'S':0,'D':0,'I':0,'N':0} for m in MODELS}
    seg_rows   = []

    for idx, row in df.iterrows():
        ref = preprocess(str(row['Human']))
        if not ref: continue
        outs    = {m: preprocess(str(row[m])) for m in MODELS}
        lattice, ci = build_lattice(ref, outs)
        seg = {'Segment':idx+1, 'Reference':str(row['Human'])[:55]}
        for m in MODELS:
            sw, lw, detail = compute_wer(ref, outs[m], lattice, ci)
            std_totals[m]['S']+=detail['S'];  std_totals[m]['D']+=detail['D']
            std_totals[m]['I']+=detail['I'];  std_totals[m]['N']+=detail['N']
            lat_totals[m]['S']+=detail['lattice_S']
            lat_totals[m]['D']+=detail['lattice_D']
            lat_totals[m]['I']+=detail['lattice_I']
            lat_totals[m]['N']+=detail['N']
            seg[f'{m}_std']=round(sw*100,1)
            seg[f'{m}_lat']=round(lw*100,1)
        seg_rows.append(seg)

    corpus_rows = []
    for m in MODELS:
        s, l = std_totals[m], lat_totals[m]
        sw = (s['S']+s['D']+s['I'])/s['N']*100 if s['N'] else 0
        lw = (l['S']+l['D']+l['I'])/l['N']*100 if l['N'] else 0
        d  = sw - lw
        if   d>5: verdict='Unfairly penalized'
        elif d>0: verdict='Minor corrections'
        else:     verdict='No change'
        corpus_rows.append({
            'Model':m, 'StdWER':round(sw,2), 'LatWER':round(lw,2),
            'Delta':round(d,2), 'Verdict':verdict,
            'S_s':s['S'],'D_s':s['D'],'I_s':s['I'],
            'S_l':l['S'],'D_l':l['D'],'I_l':l['I'], 'N':s['N'],
        })
    return corpus_rows, seg_rows


corpus_rows, seg_rows = run_evaluation(df_raw)

INTERP = {
    'Model H': 'Very accurate. Only minor Devanagari variants.',
    'Model i': 'Punctuation-only diffs = 0% WER. Gold standard.',
    'Model k': 'Roman script loanwords + punctuation, both valid conventions.',
    'Model l': 'Spelling variants (nukta, chandrabindu) penalized unfairly.',
    'Model m': 'Mix: genuine errors (Urdu script seg 46) + phonetic variants.',
    'Model n': 'Valid spelling variants (waw/yaa forms) penalized.',
}
print('FINAL RESULTS')
print(f'{"Model":<12} {"Std WER%":>10} {"Lat WER%":>12} {"Reduction":>10}  Verdict')
print('='*65)
for r in corpus_rows:
    tag = '🔴' if r['Delta']>5 else ('🟡' if r['Delta']>0 else '🟢')
    print(f'{r["Model"]:<12} {r["StdWER"]:>10.2f} {r["LatWER"]:>12.2f} '
          f'{r["Delta"]:>10.2f}  {tag} {r["Verdict"]}')
print()
for r in corpus_rows:
    print(f'  {r["Model"]}: {INTERP[r["Model"]]}')

FINAL RESULTS
Model          Std WER%     Lat WER%  Reduction  Verdict
Model H            2.81         1.83       0.98  🟡 Minor corrections
Model i            0.00         0.00       0.00  🟢 No change
Model k            5.38         3.67       1.71  🟡 Minor corrections
Model l            8.19         4.03       4.16  🟡 Minor corrections
Model m           14.30         7.33       6.97  🔴 Unfairly penalized
Model n            8.44         3.67       4.77  🟡 Minor corrections

  Model H: Very accurate. Only minor Devanagari variants.
  Model i: Punctuation-only diffs = 0% WER. Gold standard.
  Model k: Roman script loanwords + punctuation, both valid conventions.
  Model l: Spelling variants (nukta, chandrabindu) penalized unfairly.
  Model m: Mix: genuine errors (Urdu script seg 46) + phonetic variants.
  Model n: Valid spelling variants (waw/yaa forms) penalized.


In [ ]:
#inspect_segment() — explore any segment
def inspect_segment(seg_num: int):
    """Show full lattice + trust decisions + per-model WER for any segment."""
    row  = df_raw.iloc[seg_num-1]
    ref  = preprocess(str(row['Human']))
    outs = {m: preprocess(str(row[m])) for m in MODELS}
    lat, ci = build_lattice(ref, outs)

    print(f'\n{"="*65}')
    print(f'Segment {seg_num}')
    print(f'Human ref : {row["Human"]}')
    for m in MODELS:
        print(f'  {m:9}: {row[m]}')
    print(f'\n{"Pos":<5} {"Ref word":<18} {"Valid alts":<28} Trust log')
    print('-'*75)
    for b in lat:
        extras = b['valid_alternatives'] - {b['ref_word']}
        log    = {w:r for w,r in b['trust_log'].items() if w!=b['ref_word']}
        fd     = ' [DEL OK]' if b['forgive_deletion'] else ''
        print(f"{b['position']:<5} {b['ref_word']:<18} {str(extras):<28} {log}{fd}")
    if ci:
        print(f'\nConsensus inserts (ref missing): {ci}')
    print(f'\n{"Model":<12} {"Std%":>8} {"Lat%":>8} {"Saved":>8}')
    print('-'*40)
    for m in MODELS:
        sw, lw, _ = compute_wer(ref, outs[m], lat, ci)
        saved = (sw-lw)*100
        flag  = ' <- unfair penalty removed' if saved>0 else ''
        print(f'{m:<12} {sw*100:>8.1f} {lw*100:>8.1f} {saved:>8.1f}{flag}')


# Interesting segments:
#  7  = सियाराम proper noun, ref protected
#  11 = हम्म filler deletion forgiven
#  13 = consensus insertion तो (ref missing word)
#  44 = एक->ठीक ref error correctly overridden
#  46 = Model m Urdu script, correctly penalized

inspect_segment(7)
inspect_segment(44)


Segment 7
Human ref : सर आपकी इच्छा जय सियाराम
  Model H  : सर आपकी इच्छा जय श्री आराम
  Model i  : सर आपकी इच्छा जय सियाराम
  Model k  : sir आपकी इच्छा। जय सिया राम!
  Model l  : सर आपकी इच्छा जय शिय राम
  Model m  : सर आपकी इच्छा जैसी आराम
  Model n  : सर आपकी इच्छा जय सी आराम

Pos   Ref word           Valid alts                   Trust log
---------------------------------------------------------------------------
0     सर                 set()                        {}
1     आपकी               set()                        {}
2     इच्छा              set()                        {}
3     जय                 set()                        {}
4     सियाराम            {'राम', 'सिया'}              {'सिया': 'compound_split', 'राम': 'compound_split'}

Model            Std%     Lat%    Saved
----------------------------------------
Model H          40.0     20.0     20.0 <- unfair penalty removed
Model i           0.0      0.0      0.0
Model k          40.0     20.0     20.0 <- unfair penalt

In [ ]:
#Export to Excel 

DARK='FF1B3A6B'; LITE='FFD6E4F7'
GREEN='FFD9F0D4'; AMBER='FFFFF3CD'; RED='FFFCE4EC'; GRAY='FFF5F5F5'
_thin=Side(style='thin',color='FFB0B0B0')
_brd=Border(left=_thin,right=_thin,top=_thin,bottom=_thin)

def _H(c,v,bg=DARK,fg='FFFFFFFF',bold=True,sz=10,lft=False):
    c.value=v; c.font=Font(bold=bold,color=fg,size=sz,name='Arial')
    c.fill=PatternFill('solid',start_color=bg)
    c.alignment=Alignment(horizontal='left' if lft else 'center',vertical='center',wrap_text=True)
    c.border=_brd

def _D(c,v,bg='FFFFFFFF',bold=False,fmt=None,lft=False):
    c.value=v; c.font=Font(bold=bold,name='Arial',size=10)
    c.fill=PatternFill('solid',start_color=bg)
    c.alignment=Alignment(horizontal='left' if lft else 'center',vertical='center',wrap_text=True)
    c.border=_brd
    if fmt: c.number_format=fmt

def _w(ws,col,width):
    ws.column_dimensions[get_column_letter(col)].width=width

VBGS={'Unfairly penalized':RED,'Minor corrections':AMBER,'No change':GREEN}
wb=Workbook()

# Sheet 1: Corpus WER
ws1=wb.active; ws1.title='1. Corpus WER'; ws1.freeze_panes='A4'
ws1.merge_cells('A1:L1')
_H(ws1['A1'],'Q4 Lattice WER | Hindi ASR | Josh Talks',sz=13)
ws1.row_dimensions[1].height=28
ws1.merge_cells('A2:L2')
_H(ws1['A2'],'8 trust rules | Model i anchor | Consensus threshold=3 | Word-level alignment',
   bg=LITE,fg='FF1B3A6B',bold=False,sz=9)
ws1.row_dimensions[2].height=16
hdrs=['Model','Std WER%','Lat WER%','Delta','S std','D std','I std','S lat','D lat','I lat','Ref Words','Verdict']
widths=[11,11,11,11,8,8,8,8,8,8,10,20]
for ci,(h,w) in enumerate(zip(hdrs,widths),1):
    _H(ws1.cell(3,ci),h); _w(ws1,ci,w)
ws1.row_dimensions[3].height=28
for ri,r in enumerate(corpus_rows,4):
    bg=VBGS.get(r['Verdict'],'FFFFFFFF')
    dbg=RED if r['Delta']>5 else (AMBER if r['Delta']>0 else GREEN)
    _D(ws1.cell(ri,1),r['Model'],bold=True,bg=bg)
    _D(ws1.cell(ri,2),r['StdWER'],fmt='0.00',bg=bg)
    _D(ws1.cell(ri,3),r['LatWER'],fmt='0.00',bg=bg)
    _D(ws1.cell(ri,4),r['Delta'],bold=True,fmt='0.00',bg=dbg)
    for ci,k in enumerate(['S_s','D_s','I_s','S_l','D_l','I_l','N'],5):
        _D(ws1.cell(ri,ci),r[k],bg=bg)
    _D(ws1.cell(ri,12),r['Verdict'],bold=True,bg=bg)
    ws1.row_dimensions[ri].height=18

# Sheet 2: Segment WER
ws2=wb.create_sheet('2. Segment WER'); ws2.freeze_panes='C3'
ncols2=2+len(MODELS)*2
ws2.merge_cells(f'A1:{get_column_letter(ncols2)}1')
_H(ws2['A1'],'Segment WER — Standard vs Lattice',sz=12)
ws2.row_dimensions[1].height=22
h2=['Seg','Reference']+[f'{m}\nStd%' for m in MODELS]+[f'{m}\nLat%' for m in MODELS]
w2=[5,46]+[9]*len(MODELS)*2
for ci,(h,w) in enumerate(zip(h2,w2),1):
    _H(ws2.cell(2,ci),h); _w(ws2,ci,w)
ws2.row_dimensions[2].height=32
for ri,seg in enumerate(seg_rows,3):
    _D(ws2.cell(ri,1),seg['Segment'])
    _D(ws2.cell(ri,2),seg['Reference'],lft=True)
    for ci,m in enumerate(MODELS,3):
        v=seg.get(f'{m}_std',0) or 0
        _D(ws2.cell(ri,ci),v,fmt='0.0',bg=RED if v>30 else (AMBER if v>10 else GREEN))
    for ci,m in enumerate(MODELS,3+len(MODELS)):
        v=seg.get(f'{m}_lat',0) or 0
        _D(ws2.cell(ri,ci),v,fmt='0.0',bg=RED if v>30 else (AMBER if v>10 else GREEN))
    ws2.row_dimensions[ri].height=14

wb.save(OUTPUT_FILE)
print(f'Done. Saved -> {OUTPUT_FILE}')

Done. Saved -> Q4_Lattice_WER_Results.xlsx


---
## Summary

| Model | Std WER% | Lattice WER% | Verdict |
|---|---|---|---|
| Model i | 0.00 | 0.00 | Gold standard |
| Model H | 2.81 | 1.83 | Minor corrections |
| Model k | 5.38 | 3.67 | Minor corrections |
| Model l | 8.19 | 4.03 | Minor corrections |
| Model n | 8.44 | 3.67 | Minor corrections |
| Model m | 14.30 | 7.33 | Unfairly penalized |

**Use `inspect_segment(N)` in Cell 10 to drill into any of the 46 segments.**